# Test desi-photometry validation

This notebook provides tests for updating the `validate-files` script.

## Import and setup

In [ ]:
import os
import glob
import re
from astropy.io import fits
specprod = os.environ['SPECPROD'] = 'nevis'
release = {'fuji': 'edr', 'guadalupe': 'dr1', 'iron': 'dr1'}
version = {'fuji': 'v2.1', 'guadalupe': 'v2.1', 'iron': 'v1.1'}
if specprod == 'nevis':
    output_dir = os.path.join(os.environ['SCRATCH'], 'desi', 'lsdr9', specprod)
else:
    output_dir = os.path.join(os.environ['DESI_ROOT'], 'vac', release[specprod], 'lsdr9-photometry', specprod, version[specprod])

## Find the files

In [ ]:
target, tractor = dict(), dict()
for k in ('observed', 'potential'):
    target[k] = glob.glob(os.path.join(output_dir, f'{k}-targets', '*.fits'))
    tractor[k] = glob.glob(os.path.join(output_dir, f'{k}-targets', 'tractorphot', '*.fits'))

### Verify that all tractor files match

In [ ]:
tractor_re = re.compile(f'tractorphot(-potential|)-nside([0-9]+)-hp([0-9]+)-{specprod}\\.fits')
for k in ('observed', 'potential'):
    assert all([tractor_re.match(os.path.basename(filename)) is not None for filename in tractor[k]])

### Verify that all target files match

In [ ]:
target_re = re.compile(f'targetphot(-potential|)(-[a-z123]+)?(-[a-z-]+)?-{specprod}\\.fits')
target_hp_re = re.compile(f'targetphot(-potential|)-nside([0-9]+)-hp([0-9]+)-([a-z123]+)([a-z-]+)?-{specprod}\\.fits')
for k in ('observed', 'potential'):
    assert all([target_re.match(os.path.basename(filename)) is not None or target_hp_re.match(os.path.basename(filename)) is not None for filename in target[k]])

## Gather data on files

In [ ]:
target_info = {'observed': dict(), 'potential': dict()}
for k in ('observed', 'potential'):
    for filename in target[k]:
        b = os.path.basename(filename)
        m = target_re.match(b)
        m_hp = target_hp_re.match(b)
        groups = m_hp.groups() if m is None else m.groups()
        with fits.open(filename, readonly=True) as hdulist:
            rows = hdulist['TARGETPHOT'].header['NAXIS2']
        target_info[k][b] = {'fullname': filename,
                             'hp': (m_hp is not None),
                             'size': os.path.getsize(filename),
                             'groups': groups,
                             'rows': rows}
target_info

In [ ]:
tractor_info = {'observed': dict(), 'potential': dict()}
for k in ('observed', 'potential'):
    for filename in tractor[k]:
        b = os.path.basename(filename)
        m = tractor_re.match(b)
        groups = m.groups()
        with fits.open(filename, readonly=True) as hdulist:
            rows = hdulist['TRACTORPHOT'].header['NAXIS2']
        tractor_info[k][b] = {'fullname': filename,
                              'hp': (m is not None),
                              'size': os.path.getsize(filename),
                              'groups': groups,
                              'rows': rows}
tractor_info

## Print results

In [ ]:
def fmt_bytes(b):
    """Convert integer bytes `b` to a formatted string.
    """
    prefix = ('', 'k', 'M', 'G', 'T', 'P', 'E', 'Z', 'Y', 'R', 'Q')
    bb = float(b)
    p = 0
    while bb > 1024.0:
        bb /= 1024.0
        p += 1
    return f"{bb:.3g} {prefix[p]}B"

In [ ]:
b = 1000
for k in range(30):
    print(k, fmt_bytes(b))
    b *= 10

In [ ]:
for k in ('observed', 'potential'):
    target_table = ['| File Name | File Size | Number of Targets | Notes |',
                    '|-----------|:---------:|:-----------------:|-------|']
    hp_totals = dict()
    for f in sorted(target_info[k].keys()):
        if target_info[k][f]['groups'][-2] is None:
            survey, program = '', ''
        else:
            if target_info[k][f]['groups'][-1] is None:
                survey = target_info[k][f]['groups'][-2].replace('-', '')
                program = ''
            else:
                survey, program = target_info[k][f]['groups'][-2].replace('-', ''), target_info[k][f]['groups'][-1].replace('-', '')
        if target_info[k][f]['hp']:
            if (survey, program) in hp_totals:
                hp_totals[(survey, program)]['size'] += target_info[k][f]['size']
                hp_totals[(survey, program)]['rows']+= target_info[k][f]['rows']
            else:
                g = target_info[k][f]['groups']
                if program:
                    name = f"targetphot{g[0]}-nside{g[1]}-hp[0-9]{{{len(g[2])}}}-{survey}-{program}-{specprod}\\.fits"
                else:
                    name = f"targetphot{g[0]}-nside{g[1]}-hp[0-9]{{{len(g[2])}}}-{survey}-{specprod}\\.fits"
                hp_totals[(survey, program)] = {'size': target_info[k][f]['size'],
                                                'rows': target_info[k][f]['rows'],
                                                'name': name}
        else:
            if survey:
                if program:
                    target_table.append(f"| {f} | {fmt_bytes(target_info[k][f]['size'])} | {target_info[k][f]['rows']:,d} | {survey}, {program} |")
                else:
                    target_table.append(f"| {f} | {fmt_bytes(target_info[k][f]['size'])} | {target_info[k][f]['rows']:,d} | {survey} |")
            else:
                target_table.append(f"| {f} | {fmt_bytes(target_info[k][f]['size'])} | {target_info[k][f]['rows']:,d} | Concatenation of all files. |")
    for survey, program in hp_totals:
        if program:
            target_table.append(f"| {hp_totals[(survey, program)]['name']} | {fmt_bytes(hp_totals[(survey, program)]['size'])} | {hp_totals[(survey, program)]['rows']:,d} | {survey}, {program} |")
        else:
            target_table.append(f"| {hp_totals[(survey, program)]['name']} | {fmt_bytes(hp_totals[(survey, program)]['size'])} | {hp_totals[(survey, program)]['rows']:,d} | {survey} |")
    target_table.append('')
    print("\n".join(target_table))

In [ ]:
for k in ('observed', 'potential'):
    target_table = ['| File Name | File Size | Number of Targets | Notes |',
                    '|-----------|:---------:|:-----------------:|-------|']
    ex = tractor_info[k][list(tractor_info[k].keys())[0]]
    generic_name = f"tractorphot{ex['groups'][0]}-nside{ex['groups'][1]}-hp[0-9]{{{len(ex['groups'][2])}}}-{specprod}\\.fits"
    total_size = sum([tractor_info[k][f]['size'] for f in tractor_info[k]])
    total_rows = sum([tractor_info[k][f]['rows'] for f in tractor_info[k]])
    target_table.append(f"| {generic_name} | {fmt_bytes(total_size)} | {total_rows:,d} | |")
    target_table.append('')
    print("\n".join(target_table))